# 原 Stage1 0bp：风险上限受限与非受限现金拖累的全窗口静态边界

只读冻结 Qlib Recorder 的819个126日窗口，区分风险上限受限日与其他日累计现金贡献。静态修复不等于可交易账户。

In [1]:
import os,sys,json,hashlib,math
from pathlib import Path
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
root=Path('/Users/rziiiii/dev/workspace');sys.path.insert(0,str(root))
card=root/'research/stage1_original_0bp_cash_layer_window_bound_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];assert plan['frozen'] and not (card/'run_state.json').exists()
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest();parent=root/src['component_card'];assert sha(parent/'summary.json')==src['component_summary_sha256']
summary=json.loads((parent/'summary.json').read_text());assert summary['recorder_id']==src['recorder_id'] and summary['experiment_id']==src['experiment_id'] and summary['nas_evidence']['sha256']==src['NAS_archive_sha256'] and summary['nas_readback']['verified']
a=parent/'qlib_recorder'/src['experiment_id']/src['recorder_id']/'artifacts/analysis/rolling126';assert sha(a)==src['rolling_artifact_sha256']
prior=json.loads((root/src['parent_all_window_card']/'result.json').read_text());assert prior['rolling126_windows']==819 and prior['original_strict_positive_windows']==255
parent_plan=json.loads((parent/'plan.json').read_text());assert sha(parent/'plan.json')==summary['plan_sha256']
state={'plan_sha256':sha(card/'plan.json'),'phase':'native_readback_attempted','source_Recorder_object_load_attempts':1,'new_model_fits':0,'new_native_accounts':0,'new_bootstraps':0,'new_Qlib_Recorders':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/parent_plan['sources']['provider_uri']),region='us',kernels=1)
with R.uri_context(str(parent/'qlib_recorder')):
 rec=R.get_recorder(recorder_id=src['recorder_id'],experiment_id=src['experiment_id']);assert rec.status=='FINISHED';w=rec.load_object('analysis/rolling126')
assert len(w)==819 and w.sessions.eq(126).all()
required=['relative_log_excess','cash_opportunity_log','cap_binding_cash_log','nonbinding_cash_log','SOXL_selection_log','Stage1_direct_fees_log','timing_and_benchmark_residual_log']
assert np.isfinite(w[required].to_numpy(float)).all()
closure=float(np.max(np.abs(w.cash_opportunity_log-w.cap_binding_cash_log-w.nonbinding_cash_log)));assert closure<1e-10
assert np.max(np.abs(w[['cash_opportunity_log','SOXL_selection_log','Stage1_direct_fees_log','timing_and_benchmark_residual_log']].sum(axis=1)-w.relative_log_excess))<1e-10
original=w.relative_log_excess;positive=original>0;negative=~positive
cap_repaired=original-w.cap_binding_cash_log.clip(upper=0);nonbinding_repaired=original-w.nonbinding_cash_log.clip(upper=0);both_repaired=cap_repaired-w.nonbinding_cash_log.clip(upper=0)
def describe(mask):
 z=w.loc[mask]
 return {'windows':len(z),'mean_cap_binding_cash_log':float(z.cap_binding_cash_log.mean()),'mean_nonbinding_cash_log':float(z.nonbinding_cash_log.mean()),'cap_binding_cash_negative_windows':int((z.cap_binding_cash_log<0).sum()),'nonbinding_cash_negative_windows':int((z.nonbinding_cash_log<0).sum()),'both_negative_windows':int(((z.cap_binding_cash_log<0)&(z.nonbinding_cash_log<0)).sum()),'mean_cap_binding_fraction':float(z.cap_binding_fraction.mean())}
full=json.loads((parent/'result.json').read_text())['full'];need=math.ceil(.7*len(w))
result={'decision':'read_only_cap_binding_cash_alone_insufficient_on_frozen_account_path','plan_sha256':sha(card/'plan.json'),'source_recorder_id':src['recorder_id'],'cash_subcomponent_closure_max':closure,'rolling126_windows':len(w),'required_positive_windows_70pct':need,'original_positive_windows':int(positive.sum()),'static_direct_negative_cash_component_repairs_positive_windows':{'cap_binding_only':int((cap_repaired>0).sum()),'nonbinding_only':int((nonbinding_repaired>0).sum()),'both_separately':int((both_repaired>0).sum())},'groups':{'all':describe(np.ones(len(w),dtype=bool)),'nonwinning':describe(negative),'winning':describe(positive)},'full_944_day_cash_components':{'cap_binding_cash_log':float(full['cap_binding_cash_log']),'nonbinding_cash_log':float(full['nonbinding_cash_log']),'cash_opportunity_log':float(full['cash_opportunity_log'])},'static_repair_is_native_account':False,'causal_policy_inference':False,'window_overlap':True,'market_requests':0,'model_fits':0,'native_accounts':0,'bootstraps':0}
assert result['original_positive_windows']==prior['original_strict_positive_windows'] and result['required_positive_windows_70pct']==prior['rolling70pct_required_strict_positive_windows']
(card/'result.json').write_text(json.dumps(result,indent=2)+'\n')
summary_out={'experiment':plan['experiment'],'decision':result['decision'],'objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','source_recorder_id':src['recorder_id'],'source_NAS_sha256':src['NAS_archive_sha256'],'new_model_fits':0,'new_native_accounts':0,'artifact_policy':plan['artifact_policy'],'limitations':['All819 windows overlap; counts are not independent trials.','A direct accounting repair freezes Stage1 holdings, risk feedback and fees; no feasible candidate account is proven.','No source feature, model, new account or SOXX hard-gate result.']};(card/'summary.json').write_text(json.dumps(summary_out,indent=2)+'\n')
state['phase']='finished';(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');print(result)

[38745:MainThread](2026-10-23 09:03:56,003) INFO - qlib.Initialization - [config.py:453] - default_conf: client.


[38745:MainThread](2026-10-23 09:03:56,004) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.


[38745:MainThread](2026-10-23 09:03:56,005) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


/Users/rziiiii/Library/Python/3.14/lib/python/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


{'decision': 'read_only_cap_binding_cash_alone_insufficient_on_frozen_account_path', 'plan_sha256': 'e700eefafc1a34b7b74005f2a101c399c07794b3f0818a33ab4f1821d0b563a9', 'source_recorder_id': '2f0392615c4c4b15928b4ada6e01ecd9', 'cash_subcomponent_closure_max': 1.1102230246251565e-16, 'rolling126_windows': 819, 'required_positive_windows_70pct': 574, 'original_positive_windows': 255, 'static_direct_negative_cash_component_repairs_positive_windows': {'cap_binding_only': 321, 'nonbinding_only': 420, 'both_separately': 588}, 'groups': {'all': {'windows': 819, 'mean_cap_binding_cash_log': -0.031016064088476503, 'mean_nonbinding_cash_log': -0.046686642050405444, 'cap_binding_cash_negative_windows': 630, 'nonbinding_cash_negative_windows': 672, 'both_negative_windows': 543, 'mean_cap_binding_fraction': 0.289474194236099}, 'nonwinning': {'windows': 564, 'mean_cap_binding_cash_log': -0.06342743744972416, 'mean_nonbinding_cash_log': -0.04959942974236836, 'cap_binding_cash_negative_windows': 537, '

# 原 Stage1 0bp：现金拖累在配置层与风险上限层均存在

只读冻结完整 Stage1 自有状态 0bp、仍扣 IBKR Tiered 费用的原生 Qlib Recorder `2f0392615c4c4b15928b4ada6e01ecd9`。819个连续126日窗口原账户仅255个胜SOXX，合同滚动70%门槛需要574个。现金机会分量严格分解为风险上限受限日与非受限日，两者的逐窗闭合误差最大约1.1e-16。零新市场请求、模型拟合、账户和bootstrap。

|原账户路径上的静态修复|胜SOXX窗口|
|---|---:|
|原始Stage1|255/819|
|只抹去受限日的负现金分量|321/819|
|只抹去非受限日的负现金分量|420/819|
|分别抹去两类负现金分量|588/819|
|合同70%最低要求|574/819|

564个未胜窗口中，受限日现金分量为负537窗，非受限日为负480窗，二者同时为负460窗。全944日累计现金机会对数拖累 -0.60032，其中受限日 -0.28701、非受限日 -0.31331。**两层都重要**：在冻结账户路径上的过度乐观静态账本里，单修其中一层也不够滚动门槛。255个原胜窗的受限日现金分量均值反而为 +0.04067，说明原账本中这部分现金在部分好窗口贡献为正；不能把亏损窗口里的受限状态直接当作放松上限的交易信号。

上述修复不是持仓、费用与风险反馈重新计算后的真实账户，也不能证明某一层的策略不可能成功。下一特征需要在决策时同时支持现金配置和风险约束的条件判断，并在完整0bp自有状态账户上保留全部年度、季度及819个窗口复核。现有期权偏斜等候选仍有历史来源/首次可用时点缺口；未运行新候选账户，SOXX稳定超额、2/5bp及独立前向门仍未通过。
